# CIC-IDS2017 Monday: NFStream extraction and daily labeling

This notebook extracts bidirectional network flows from the original CIC-IDS2017 Monday PCAP capture using NFStream. It cleans the extracted records, creates the timestamp used by the original labeling procedure, applies the day-specific attack rules, assigns the remaining flows to the benign class, removes duplicate rows, and exports the daily CSV consumed by the GenIDS-CIC17 consolidation notebook.

Only the input and output paths in the configuration cell should be changed. The original PCAP capture is not distributed with this repository.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_FILE = Path("/path/to/cic-ids2017/pcaps/01_monday.pcap")
OUTPUT_DIR = Path("/path/to/output/nfstream_daily_csv")
OUTPUT_FILE = OUTPUT_DIR / "01_nfstream_monday.csv"

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 20
STATISTICAL_ANALYSIS = True
DECODE_TUNNELS = True
BPF_FILTER = 'ip'
TIMEZONE = "America/Moncton"

DROP_COLUMNS = [
    "content_type",
    "user_agent",
    "server_fingerprint",
    "client_fingerprint",
    "requested_server_name",
]

## 2. Imports and helper functions

In [ ]:
import pandas as pd
import pytz
import nfstream
from nfstream import NFStreamer


def validate_input(pcap_file):
    if not pcap_file.is_file():
        raise FileNotFoundError(f"PCAP file not found: {pcap_file}")


def extract_flows(pcap_file):
    return NFStreamer(
        source=str(pcap_file),
        idle_timeout=IDLE_TIMEOUT,
        active_timeout=ACTIVE_TIMEOUT,
        statistical_analysis=STATISTICAL_ANALYSIS,
        decode_tunnels=DECODE_TUNNELS,
        bpf_filter=BPF_FILTER,
    ).to_pandas()


def prepare_flows(frame):
    prepared = frame.drop(columns=DROP_COLUMNS, errors="ignore").dropna().copy()
    if "src2dst_first_seen_ms" not in prepared.columns:
        raise ValueError("NFStream output is missing the src2dst_first_seen_ms column.")
    timestamps = pd.to_datetime(
        prepared["src2dst_first_seen_ms"], unit="ms", utc=True
    ).dt.tz_convert(pytz.timezone(TIMEZONE))
    prepared["Timestamp"] = timestamps.dt.strftime("%d/%m/%Y %I:%M")
    prepared["binary"] = ""
    prepared["multiclass"] = ""
    return prepared.reset_index(drop=True)


def class_summary(frame, column):
    return pd.DataFrame({
        "count": frame[column].value_counts(),
        "percentage": frame[column].value_counts(normalize=True).mul(100).round(2),
    })


def save_daily_flows(frame, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_file, index=False)
    if not output_file.is_file():
        raise OSError(f"The output file was not created: {output_file}")

## 3. Validate the input and extract NFStream flows

In [ ]:
print(f"NFStream version: {nfstream.__version__}")
validate_input(PCAP_FILE)

raw_flows = extract_flows(PCAP_FILE)
print(f"Extracted flows: {len(raw_flows):,}")
print(f"Extracted columns: {raw_flows.shape[1]}")

## 4. Clean and prepare the extracted flows

In [ ]:
daily_flows = prepare_flows(raw_flows)

print(f"Flows after cleaning: {len(daily_flows):,}")
print(f"Rows removed during cleaning: {len(raw_flows) - len(daily_flows):,}")

## 5. Apply the Monday labeling rules

In [ ]:
df = daily_flows

df.loc[(df['binary'] == ''), df.binary.name] = 'benign'
df.loc[(df['multiclass'] == ''), df.multiclass.name] = 'benign'

daily_flows = df

## 6. Remove duplicates and summarize the daily dataset

In [ ]:
rows_before = len(daily_flows)
daily_flows = daily_flows.drop_duplicates().reset_index(drop=True)

unlabeled = daily_flows[["binary", "multiclass"]].eq("").any(axis=1).sum()
if unlabeled:
    raise ValueError(f"Found {unlabeled} flows without complete labels.")

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Rows after duplicate removal:  {len(daily_flows):,}")
display(class_summary(daily_flows, "binary"))
display(class_summary(daily_flows, "multiclass"))

## 7. Export the daily flow file

In [ ]:
save_daily_flows(daily_flows, OUTPUT_FILE)

print(f"Daily dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final shape: {daily_flows.shape}")
display(daily_flows.head())